In [ ]:
!pip install faiss-cpu sentence-transformers pypdf fpdf langchain langchain-community

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=4d3cea74fd06f6fc0c01bf1d1ba38322b0d1471222e5012040dbd4823ab5333f
  Stored in directory: /root/.cache/pip/wheels/aa/da/11/a3189f34ddc13c26a2d0f329eac46b728c7f31c39e4dc26243
Successfully built fpdf
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not cur

In [ ]:
# import faiss, sentence transformer
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np
from fpdf import FPDF
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

In [ ]:
from pypdf import PdfReader

# Load your resume PDF
pdf_reader = PdfReader("/content/Aditya Sisodiya Resume-1.pdf")

raw_text = ""

for page_number, page in enumerate(pdf_reader.pages):
    text = page.extract_text()

    if text:
        raw_text += text + "\n"

print("Resume loaded successfully!")
print(raw_text)

FileNotFoundError: [Errno 2] No such file or directory: '/content/Aditya Sisodiya Resume-1.pdf'

In [ ]:
print("Number of characters:", len(raw_text))
print("\nResume text:")
print(raw_text[:3000])

In [ ]:
!pip install -q langchain-huggingface langchain-text-splitters

In [ ]:
# import libraries

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

In [ ]:
from transformers import pipeline
import torch

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)


In [ ]:
LLM = HuggingFacePipeline(pipeline=generator)

In [ ]:
from langchain_core.documents import Document

document = Document(
    page_content=raw_text,
    metadata={
        "source": "Aditya Sisodiya Resume-1.pdf"
    }
)

print(document)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=250
)


In [ ]:

chunks = text_splitter.split_documents([document])

print("Number of chunks:", len(chunks))

In [ ]:
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content)

# Embeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector database created successfully!")

# retriver

In [ ]:
retriver = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
query = "What are my technical skills?"

docs = retriver.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

### Build RAG chain LLM|RAG  
# chat prompt templet

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a resume question-answering assistant.

Answer the user's question using ONLY the resume context below.

Resume Context:
{context}

If the answer is present in the context, answer it directly.
If the answer is not present, say:
"I don't have enough information in the resume."
"""
    ),

    MessagesPlaceholder(variable_name="chat_history"),

    ("human", "{question}")
])

In [ ]:
  # combine the chunk to provide full context to llm
  def combine_chunks(chunks):
    return "\n\n".join(chunk.page_content for chunk in chunks)


In [ ]:
def get_context(input_dict):
    question = input_dict["question"]
    docs = retriver.invoke(question)
    return combine_chunks(docs)

RAGPIPE = {
    "context": get_context,
    "question": lambda x: x["question"],
    "chat_history": lambda x: x["chat_history"]
}

chain = RAGPIPE | prompt | LLM | StrOutputParser()

# add Memory


In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]

In [ ]:
# existing chain ko memory ke saath wrap karo

In [ ]:

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

In [ ]:
response = chain.invoke({
    "question": "What projects have I worked on?",
    "chat_history": []
})

print(response)

In [ ]:
response = chain_with_memory.invoke(
    {
        "question": "Which technologies did I use in that project?"
    },
    config={
        "configurable": {
            "session_id": "user_1"
        }
    }
)

print(response)

In [ ]:
chunking the data -> embeddings -> vector database -> retrivel the context -> design the prompt templet -> combine_chunks() (Context+question+chathistory )
